# Lección 15.4 — Sesgo y equidad: cómo los modelos heredan prejuicios

> **Módulo 15 · ML en producción y responsabilidad** · Nivel 5 — Héroe
> ⏱️ Tiempo estimado: 90–110 min (lección con ejercicios intercalados ≈ 65 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 4.2 (precisión, recall, matriz de confusión), Lección 4.3 (umbrales), Lección 15.2

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar de dónde viene el **sesgo** de un modelo: datos históricos, representación, medición y variables *proxy*.
- **Medir** la equidad de un modelo calculando métricas **por grupo**: tasa de selección, TPR, FPR, precisión e **impacto dispar**.
- Entender por qué **quitar la variable sensible no basta** y por qué no se pueden cumplir todas las definiciones de equidad a la vez.
- Comparar estrategias de **mitigación** (mejores etiquetas, quitar proxies, umbrales por grupo) y sus costos.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

plt.rcParams["figure.figsize"] = (8, 4)
print("✅ Todo listo")

---
## 1. 📖 La intuición: el espejo que amplifica 🪞

Un modelo de ML aprende **imitando el pasado**. Si el pasado fue injusto, el modelo aprende esa injusticia… y la aplica a miles de personas, en silencio y con apariencia de objetividad ("lo decidió el algoritmo").

> 📰 **Caso real (Amazon, 2018):** un sistema experimental para filtrar currículums aprendió de 10 años de contrataciones, en su mayoría de hombres. Empezó a **penalizar** CVs que contenían la palabra *women's* (por ejemplo, "capitana del club de ajedrez **femenino**"). Amazon abandonó el proyecto.

Nadie programó "discrimina a las mujeres". El modelo lo **aprendió de los datos**.

### ✋ Ejercicio 1 — Calentamiento
Un banco entrena un modelo de crédito con las decisiones que tomaron sus analistas en los últimos 20 años. Durante parte de ese tiempo, a ciertos barrios casi no se les prestaba. ¿Qué crees que aprenderá el modelo sobre esos barrios? ¿Por qué es difícil descubrirlo mirando solo la exactitud?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Aprenderá que vivir en esos barrios "predice" un rechazo, porque eso es lo que decían las etiquetas. Y la **exactitud** será alta: el modelo imita muy bien las decisiones pasadas. El problema es que la etiqueta medía **lo que decidieron los analistas**, no **si la persona habría pagado**. Un modelo puede ser exacto y a la vez injusto: hay que medir **por grupo** y preguntarse **qué mide la etiqueta**.
</details>

---
## 2. 📖 El concepto: de dónde viene el sesgo

| Fuente | Qué pasa | Ejemplo |
|---|---|---|
| **Sesgo histórico** | Los datos reflejan decisiones o desigualdades pasadas | Contrataciones históricas mayoritariamente de un grupo |
| **Sesgo de representación** | Un grupo aparece poco en los datos | Reconocimiento facial entrenado con pocas caras de piel oscura (*Gender Shades*, 2018) |
| **Sesgo de medición** | La etiqueta o una variable mide otra cosa en cada grupo | Usar "arrestos" como medida de "delitos": depende de dónde patrulla la policía |
| **Variables *proxy*** | Una variable "inocente" está muy correlacionada con la sensible | Código postal ↔ origen étnico; huecos en el CV ↔ maternidad |
| **Sesgo de uso** | El modelo se usa para algo distinto o con otra población | Un modelo entrenado en un país, usado en otro |

### ✋ Ejercicio 2 — Clasifica la fuente
1. Un sistema de salud en EE. UU. usaba el **gasto médico pasado** para estimar quién necesitaba más cuidados. Los pacientes negros, con el mismo nivel de enfermedad, gastaban menos (por menor acceso).
2. Un asistente de voz entiende peor a personas con acento, porque casi no había grabaciones con acentos en el entrenamiento.
3. Un modelo de seguros no usa el sexo, pero sí el "tipo de revista a la que está suscrito".

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Sesgo de medición** (y histórico): el gasto **no mide** lo mismo en ambos grupos. Es el caso de Obermeyer et al. (*Science*, 2019); al cambiar la etiqueta de "gasto" a "enfermedad", el sesgo se redujo muchísimo.
2. **Sesgo de representación.**
3. **Variable proxy**: la suscripción puede revelar el sexo (u otra característica) sin nombrarlo.
</details>

---
## 3. 📖 Cómo se mide: métricas por grupo

Vamos a auditar un **filtro de currículums**. Simulamos 10,000 candidatos de dos grupos, **A** (65 %) y **B** (35 %). Puede ser cualquier par de grupos protegidos (mujeres/hombres, un origen étnico, una zona…). Lo importante de la simulación:

- La **capacidad real** se distribuye **igual** en los dos grupos (lo sabemos porque la simulamos nosotros).
- La etiqueta de entrenamiento es **"fue contratado en el pasado"**, decidida por evaluadores que, a igual capacidad, puntuaban más bajo al grupo B.
- El grupo B tiene, en promedio, **menos años de experiencia** (menos oportunidades) y **más meses de pausa laboral** (por ejemplo, cuidados), y vive más en ciertas **zonas**.
- El modelo **no recibe** la columna `grupo`.

In [ ]:
def generar_candidatos(n=10_000, sesgo_evaluadores=0.7, semilla=0):
    rng = np.random.default_rng(semilla)
    es_b = (rng.random(n) < 0.35).astype(int)
    capacidad = rng.normal(0, 1, n)                               # ¡igual en ambos grupos!
    df = pd.DataFrame({
        "grupo": np.where(es_b == 1, "B", "A"),
        "puntaje_prueba": np.round(70 + 10 * (capacidad + rng.normal(0, 0.6, n))),
        "anios_experiencia": np.clip(np.round(5 + 2 * capacidad + rng.normal(0, 2, n) - 1.5 * es_b), 0, None),
        "brecha_laboral_meses": np.round(rng.exponential(np.where(es_b == 1, 10, 3))),
        "zona": np.where(es_b == 1, rng.choice([1, 2, 3, 4, 5], n, p=[.05, .1, .15, .3, .4]),
                                    rng.choice([1, 2, 3, 4, 5], n, p=[.3, .3, .2, .1, .1])),
    })
    evaluacion = capacidad + rng.normal(0, 0.5, n) - sesgo_evaluadores * es_b   # evaluadores sesgados
    df["contratado_historico"] = (evaluacion > 0.5).astype(int)                 # ← la etiqueta que tenemos
    df["realmente_calificado"] = (capacidad > 0.5).astype(int)                  # ← la verdad (solo en simulación)
    return df

candidatos = generar_candidatos()
print(candidatos.groupby("grupo")[["contratado_historico", "realmente_calificado"]].mean().round(3))

### 🔮 Ejercicio 3 — ¿Qué aprenderá?
En los datos, los dos grupos están **igual de calificados** (~31 %), pero en el pasado se contrató al ~32 % del grupo A y solo al ~14 % del B. Entrenamos una regresión logística **sin** la columna `grupo`. ¿Qué % de cada grupo crees que seleccionará el modelo?

✍️ *Mi predicción:*

In [ ]:
VARIABLES = ["puntaje_prueba", "anios_experiencia", "brecha_laboral_meses", "zona"]
entren, prueba = train_test_split(candidatos, test_size=0.3, random_state=0, stratify=candidatos.grupo)

modelo = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
modelo.fit(entren[VARIABLES], entren.contratado_historico)
prueba = prueba.copy()
prueba["seleccionado"] = (modelo.predict_proba(prueba[VARIABLES])[:, 1] > 0.5).astype(int)

print("Coeficientes:", dict(zip(VARIABLES, modelo[-1].coef_[0].round(2))))
print("Exactitud global vs. etiqueta histórica:", round((prueba.seleccionado == prueba.contratado_historico).mean(), 3))
print("\n% seleccionado por grupo:", prueba.groupby("grupo").seleccionado.mean().round(3).to_dict())

<details><summary><b>👀 Ver respuesta</b></summary>

El modelo selecciona aproximadamente al **25 % del grupo A** y al **13 % del grupo B**: casi la mitad, aunque la capacidad real es la misma. Con una exactitud global de ≈ 0.84, nadie sospecharía nada mirando solo la métrica global.

Mira los coeficientes: además de la prueba y la experiencia, el modelo usa la **brecha laboral** y la **zona** con signo negativo. No predicen la capacidad; predicen **a qué grupo perteneces**, y el grupo predecía la etiqueta sesgada.
</details>

### 📖 Las métricas de equidad
Todas se calculan **por grupo** y se comparan:

| Métrica | Pregunta | Definición de equidad asociada |
|---|---|---|
| **Tasa de selección** | ¿A qué % de cada grupo selecciona? | **Paridad demográfica**: tasas iguales |
| **TPR** (*recall*) | De los que **sí** merecían ser seleccionados, ¿a qué % selecciona? | **Igualdad de oportunidades**: TPR iguales |
| **FPR** | De los que **no** lo merecían, ¿a qué % selecciona por error? | **Probabilidades igualadas** (*equalized odds*): TPR **y** FPR iguales |
| **Precisión** | De los seleccionados, ¿qué % lo merecía? | **Paridad predictiva** |

**Impacto dispar** (*disparate impact*) = tasa de selección del grupo desfavorecido ÷ la del favorecido. En EE. UU., la **regla de los cuatro quintos** considera alarma un valor **< 0.8**.

In [ ]:
def auditar(df, pred="seleccionado", verdad="realmente_calificado"):
    filas = {}
    for g, d in df.groupby("grupo"):
        filas[g] = {"n": len(d), "tasa selección": d[pred].mean(),
                    "TPR": d.loc[d[verdad] == 1, pred].mean(), "FPR": d.loc[d[verdad] == 0, pred].mean(),
                    "precisión": d.loc[d[pred] == 1, verdad].mean(), "exactitud": (d[pred] == d[verdad]).mean()}
    tabla = pd.DataFrame(filas).T
    impacto = tabla.loc["B", "tasa selección"] / tabla.loc["A", "tasa selección"]
    print(f"Impacto dispar (B/A) = {impacto:.2f} {'🚨 < 0.8' if impacto < 0.8 else '✅'}")
    return tabla.round(3)

### 🔮 Ejercicio 4 — Contra la verdad
Medimos el modelo contra la **capacidad real** (`realmente_calificado`), algo que en la vida real casi nunca tendremos, pero que la simulación nos regala. De cada 100 candidatos **realmente calificados** del grupo B, ¿cuántos crees que selecciona? ¿Y del A?

✍️ *Mi predicción:*

In [ ]:
auditar(prueba)

<details><summary><b>👀 Ver respuesta</b></summary>

- **Impacto dispar ≈ 0.54**: muy por debajo de 0.8 → 🚨.
- **TPR:** el modelo selecciona a ≈ **65 de cada 100** candidatos calificados del grupo A, pero solo a ≈ **41 de cada 100** del grupo B. A igual capacidad, una persona del grupo B tiene muchas menos oportunidades.
- La **precisión** del grupo B es más alta (≈ 0.94 vs. 0.84): a los pocos de B que selecciona, les exige más. Es lo que pasa cuando un grupo tiene que ser "el doble de bueno".

Curiosidad: la **exactitud** del grupo B no es baja (≈ 0.81). Las métricas globales y la exactitud esconden el problema; las tasas **por grupo** lo revelan.
</details>

---
## 4. 📖 ¿Qué pasa si simplemente no usamos la variable sensible?

Es la idea más común ("equidad por desconocimiento", *fairness through unawareness*)… y ya vimos que **no funciona**: nuestro modelo nunca vio `grupo`. ¿Por qué? Porque las otras variables lo **delatan**.

### 🔮 Ejercicio 5 — ¿Se puede adivinar el grupo?
Entrenamos un modelo que intenta **adivinar el grupo** usando solo las 4 variables del filtro. Si fueran neutrales, su AUC sería ≈ 0.5 (azar). ¿Qué AUC esperas?

✍️ *Mi predicción:*

In [ ]:
adivino = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
adivino.fit(entren[VARIABLES], entren.grupo == "B")
print(f"AUC adivinando el grupo con las 4 variables: {roc_auc_score(prueba.grupo == 'B', adivino.predict_proba(prueba[VARIABLES])[:, 1]):.3f}")

for v in VARIABLES:
    a = make_pipeline(StandardScaler(), LogisticRegression()).fit(entren[[v]], entren.grupo == "B")
    print(f"   solo {v:<22} → AUC {roc_auc_score(prueba.grupo == 'B', a.predict_proba(prueba[[v]])[:, 1]):.3f}")

<details><summary><b>👀 Ver respuesta</b></summary>

Con las 4 variables se adivina el grupo con un AUC ≈ **0.89**: muy por encima del azar. Las variables más delatoras son la **zona** y la **brecha laboral** (proxies); el puntaje de la prueba casi no dice nada del grupo (≈ 0.5). Los años de experiencia algo (≈ 0.65), porque el grupo B tuvo menos oportunidades.

Moraleja: **quitar la columna sensible no elimina la información sensible**. Paradójicamente, para **medir** la equidad sí necesitas conocer el grupo de cada persona.
</details>

### ✏️ Ejercicio 6 — Quitar los proxies
Quitemos también `brecha_laboral_meses` y `zona`. ¿Desaparece el problema?

✍️ *Mi predicción:*

In [ ]:
SIN_PROXIES = ["puntaje_prueba", "anios_experiencia"]     # ✏️ prueba otras combinaciones
m2 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(entren[SIN_PROXIES], entren.contratado_historico)
p2 = prueba.copy(); p2["seleccionado"] = (m2.predict_proba(p2[SIN_PROXIES])[:, 1] > 0.5).astype(int)
auditar(p2)

<details><summary><b>👀 Ver respuesta</b></summary>

**Mejora, pero no desaparece**: el impacto dispar sube (≈ 0.69) y el TPR del grupo B pasa de ≈ 0.41 a ≈ 0.48, pero sigue lejos del grupo A (≈ 0.63). Quedan dos causas:
1. La **etiqueta** sigue sesgada: el modelo sigue aprendiendo a imitar a evaluadores injustos.
2. Los **años de experiencia** cargan una desigualdad estructural (menos oportunidades para B).

Quitar proxies a ciegas también tiene costos: a veces una variable correlacionada con el grupo **sí** es relevante para la tarea. Hay que decidirlo con criterio, no automáticamente.
</details>

### 📖 La raíz: la etiqueta
¿Y si en lugar de "fue contratado en el pasado" tuviéramos una etiqueta que mida de verdad lo que queremos (por ejemplo, el **desempeño real** en el trabajo, evaluado de forma justa)? En la simulación podemos probarlo:

In [ ]:
m3 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(entren[VARIABLES], entren.realmente_calificado)
p3 = prueba.copy(); p3["seleccionado"] = (m3.predict_proba(p3[VARIABLES])[:, 1] > 0.5).astype(int)
print("Coeficientes:", dict(zip(VARIABLES, m3[-1].coef_[0].round(2))))
auditar(p3)

### ✋ Ejercicio 7 — Compara las tres versiones
Compara el TPR de los grupos en los tres modelos: original, sin proxies, y con buena etiqueta. ¿Cuál cierra la brecha? ¿Qué le pasó al coeficiente de `zona` y `brecha_laboral_meses`?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Con la **buena etiqueta** los TPR quedan prácticamente iguales (≈ 0.76 y ≈ 0.77) y el impacto dispar sube a ≈ 0.90, **aunque usa las 4 variables**. Los coeficientes de `zona` y `brecha_laboral_meses` quedan pequeños (≈ 0.1–0.2) y ya no negativos: como la etiqueta ya no depende del grupo, adivinar el grupo **ya no sirve** para predecirla.

La lección más importante de esta sección: **el sesgo suele estar en la etiqueta**. Antes de pelearte con las variables, pregúntate: *¿qué mide realmente lo que intento predecir?* (Es exactamente lo que encontró el estudio de Obermeyer del Ejercicio 2.) La selección de B sigue siendo algo menor que la de A, porque los años de experiencia arrastran la desigualdad de oportunidades: eso ya no es un problema del modelo, sino del mundo.
</details>

---
## 5. 📖 No se puede tener todo: el teorema de imposibilidad

Sería ideal cumplir **todas** las definiciones de equidad a la vez. Pero se demostró (Kleinberg et al., 2016; Chouldechova, 2017) que **si la proporción real de positivos difiere entre grupos**, un modelo **no puede** ser a la vez bien calibrado (paridad predictiva) y tener TPR y FPR iguales, salvo que sea perfecto.

Este fue el corazón del debate sobre **COMPAS** (2016), un sistema de riesgo de reincidencia en EE. UU.: ProPublica mostró que tenía un FPR mucho mayor para personas negras; la empresa respondió que estaba igual de calibrado para ambos grupos. **Ambos tenían razón**: medían definiciones distintas, incompatibles entre sí.

### 🔮 Ejercicio 8 — Un modelo perfectamente calibrado
Simulamos un puntaje **perfectamente calibrado** (si dice 70 %, el 70 % de esas personas son positivas) en dos grupos con distinta proporción de positivos (50 % y 20 %). Con el mismo umbral de 0.5, ¿tendrán el mismo FPR?

✍️ *Mi predicción:*

In [ ]:
rng = np.random.default_rng(0)
for grupo, (a, b) in {"X (50 % positivos)": (2, 2), "Y (20 % positivos)": (1, 4)}.items():
    puntaje = rng.beta(a, b, 100_000)                      # probabilidad "real" de cada persona
    positivo = rng.random(100_000) < puntaje               # calibrado por construcción
    pred = puntaje > 0.5
    print(f"{grupo}: tasa real {positivo.mean():.2f} · TPR {pred[positivo].mean():.2f} · "
          f"FPR {pred[~positivo].mean():.2f} · precisión {positivo[pred].mean():.2f}")

<details><summary><b>👀 Ver respuesta</b></summary>

**No.** Ambos grupos están perfectamente calibrados, pero el grupo X tiene un FPR muchísimo mayor (≈ 0.32 vs. ≈ 0.03) y también un TPR mayor (≈ 0.68 vs. ≈ 0.19), y la precisión tampoco coincide (≈ 0.68 vs. ≈ 0.61). Con proporciones reales distintas, **igualar unas métricas desiguala otras**.

Por eso elegir la métrica de equidad es una **decisión de valores**, no técnica: ¿qué error es más grave en este contexto, y para quién? En contratación, quizá importa más la igualdad de oportunidades (TPR); en justicia penal, quizá los falsos positivos (FPR). Esa conversación debe incluir a los afectados, a expertos legales y al negocio.
</details>

---
## 6. 📖 Mitigar: las perillas de la equidad

| Etapa | Técnica | Qué hace | Costo / riesgo |
|---|---|---|---|
| **Antes** de entrenar | Mejorar la etiqueta | Medir lo que de verdad importa | Caro, lento, pero ataca la raíz |
| | Más datos del grupo poco representado | Reducir el sesgo de representación | Requiere recolectar |
| | Revisar proxies | Quitar variables que solo "delatan" el grupo | Puede quitar información legítima |
| **Durante** el entrenamiento | Restricciones de equidad (p. ej., `fairlearn`) | Entrenar optimizando exactitud **y** una métrica de equidad | Algo menos de exactitud |
| **Después** | Umbrales por grupo | Elegir un umbral distinto por grupo para igualar TPR o tasas | Usa la variable sensible al decidir: en algunos países y contextos es **ilegal** |

Veamos la última. Volviendo al modelo original, buscaremos un umbral para el grupo B que iguale su **tasa de selección** con la de A.

### ✏️ Ejercicio 9 — El umbral del grupo B
Cambia `UMBRAL_B` (el de A se queda en 0.5). Prueba 0.5, 0.35, 0.3 y 0.25. ¿Con cuál el impacto dispar supera 0.8? ¿Qué pasa con la precisión de B?

✍️ *Mis observaciones:*

In [ ]:
UMBRAL_A, UMBRAL_B = 0.5, 0.3      # ✏️ prueba UMBRAL_B = 0.5, 0.35, 0.3, 0.25

p4 = prueba.copy()
prob = modelo.predict_proba(p4[VARIABLES])[:, 1]
p4["seleccionado"] = (prob > np.where(p4.grupo == "B", UMBRAL_B, UMBRAL_A)).astype(int)
auditar(p4)

<details><summary><b>👀 Ver respuesta</b></summary>

| UMBRAL_B | Impacto dispar | TPR de B (A = 0.65) | Precisión de B (A = 0.84) |
|---|---|---|---|
| 0.5 | 0.54 🚨 | 0.41 | 0.94 |
| 0.35 | **0.84** ✅ | 0.60 | 0.89 |
| 0.3 | 1.01 | 0.70 | 0.87 |
| 0.25 | 1.17 | 0.75 | 0.79 |

Con **0.35** ya se supera 0.8; con **0.3** las tasas de selección quedan iguales (y el TPR de B incluso supera al de A); con **0.25** el grupo B pasa a ser seleccionado **más** que A. La precisión de B baja a medida que bajas su umbral, pero hasta 0.3 sigue por encima de la de A: los candidatos de B que el modelo "rescata" son, en su mayoría, calificados.

Funciona como "parche", pero: (1) no corrige la etiqueta sesgada; (2) usa el grupo explícitamente al decidir, lo que la ley prohíbe en ciertos contextos (por ejemplo, en préstamos en EE. UU.); (3) hay que reajustarlo si cambian los datos. Por eso es mejor atacar la causa (la etiqueta) siempre que se pueda.
</details>

> 🛠️ **Para Colab:** la librería `fairlearn` (Microsoft) automatiza todo esto: `MetricFrame` calcula las métricas por grupo y `ThresholdOptimizer` / `ExponentiatedGradient` mitigan durante o después del entrenamiento.
> ```python
> # !pip install fairlearn
> from fairlearn.metrics import MetricFrame, selection_rate, true_positive_rate
> MetricFrame(metrics={"selección": selection_rate, "TPR": true_positive_rate},
>             y_true=y, y_pred=pred, sensitive_features=grupo).by_group
> ```

---
## 7. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| "No usamos el género, así que el modelo es justo" | Proxies | Medir métricas por grupo; intentar adivinar el grupo con las variables |
| Buena exactitud global, quejas de un grupo | Métricas agregadas ocultan diferencias | Auditar **por grupo** (y por combinaciones de grupos: interseccionalidad) |
| Se "corrige" una métrica y otra empeora | Teorema de imposibilidad | Elegir conscientemente la métrica según el contexto y documentarlo |
| El modelo imita perfectamente decisiones pasadas injustas | Etiqueta sesgada | Revisar qué mide la etiqueta; buscar una mejor |
| Métricas muy inestables para un grupo | Pocos datos de ese grupo | Intervalos de confianza; más datos; no sacar conclusiones con 30 casos |
| El modelo era justo al desplegarlo y ya no | Deriva (Lección 15.2) por grupo | Incluir métricas de equidad en el monitoreo |

### 🔮 Ejercicio 10 — Diagnóstico
Un equipo reporta: *"Nuestro modelo de detección de fraude tiene 95 % de exactitud tanto en clientes jóvenes como mayores, así que es justo"*. ¿Qué preguntarías antes de aceptarlo?

✍️ *Mis preguntas:*

<details><summary><b>👀 Ver respuesta</b></summary>

- ¿Cuál es el **FPR** por grupo? Si a los jóvenes se les bloquea la tarjeta por error el doble de veces, es un trato desigual aunque la exactitud sea igual.
- ¿Y el **TPR**? ¿Detecta igual el fraude contra las personas mayores (que suelen ser más vulnerables)?
- ¿Cuál es la **proporción de fraude** en cada grupo? Con mucho desbalance, 95 % de exactitud puede no decir nada (Lección 4.2).
- ¿Cuántos casos hay de cada grupo? ¿Hay subgrupos (jóvenes de zonas rurales) con resultados peores?
</details>

---
## 8. 📋 Ficha: auditoría de equidad en 6 pasos

| Paso | Qué hacer |
|---|---|
| 1 | Definir los **grupos** a proteger (y conseguir esa información para auditar, aunque no se use para decidir) |
| 2 | Preguntarse **qué mide la etiqueta** y si está sesgada |
| 3 | Calcular métricas **por grupo**: tasa de selección, TPR, FPR, precisión, impacto dispar |
| 4 | Buscar **proxies**: ¿se puede adivinar el grupo con las variables? |
| 5 | Elegir, con los afectados y el área legal, la **definición de equidad** relevante |
| 6 | Mitigar (etiqueta → datos → entrenamiento → umbrales), **documentar** y **monitorear** en el tiempo |

## 🧠 Resumen en 5 puntos
1. Los modelos aprenden del pasado: si el pasado fue injusto, **heredan y escalan** el prejuicio.
2. El sesgo viene de datos históricos, representación, **medición (etiquetas)** y **proxies**.
3. Se detecta midiendo **por grupo**: tasa de selección, TPR, FPR, precisión e impacto dispar (regla 4/5).
4. **Quitar la variable sensible no basta**, y **no se pueden** cumplir todas las definiciones de equidad a la vez: elegir es una decisión de valores.
5. La mitigación más efectiva suele ser **arreglar la etiqueta**; los umbrales por grupo son un parche con implicaciones legales.

---
## 9. 🏋️ Ejercicios finales

### 9.1 Repaso rápido (opción múltiple)

**1.** Tasa de selección: grupo A 40 %, grupo B 28 %. ¿Impacto dispar y diagnóstico con la regla 4/5?
- a) 0.70, alarma · b) 1.43, sin alarma · c) 0.12, alarma · d) 0.70, sin alarma

**2.** "De las personas que sí merecían el crédito, ¿a qué % se lo aprobó el modelo en cada grupo?" Esa es la métrica de…
- a) Paridad demográfica · b) Igualdad de oportunidades (TPR) · c) Precisión · d) Exactitud

**3.** El modelo no usa la variable "origen", pero sí el código postal. Esto puede ser un problema porque…
- a) El código postal es un número · b) Puede funcionar como proxy del origen · c) Ralentiza el modelo · d) No es un problema

**4.** Según el teorema de imposibilidad, si las tasas reales difieren entre grupos…
- a) Todo modelo es injusto · b) No se pueden igualar a la vez calibración, TPR y FPR (salvo un modelo perfecto) · c) Hay que quitar el grupo · d) Basta con más datos

**5.** ¿Cuál suele ser la mitigación que ataca la **raíz** del sesgo histórico?
- a) Umbrales por grupo · b) Mejorar lo que mide la etiqueta · c) Usar un modelo más complejo · d) Subir el umbral para todos

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **a)** 28/40 = 0.70 < 0.8. 2. **b)** 3. **b)** 4. **b)** 5. **b)**
</details>

### 9.2 Preguntas para pensar (sin ejecutar)

**A.** Para auditar la equidad necesitas saber el grupo de cada persona, pero la ley de protección de datos limita recolectar datos sensibles. ¿Cómo resolverías esa tensión?

**B. Caso de análisis 💼** Un hospital usa un modelo para priorizar pacientes para un programa de cuidados especiales. Elegir la métrica de equidad: ¿TPR iguales (que los enfermos de todos los grupos tengan la misma probabilidad de entrar) o tasas de selección iguales? Argumenta.

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** Opciones: recolectar el dato sensible **solo para auditar**, de forma voluntaria, separada y protegida (acceso restringido, nunca como entrada al modelo); auditar con una **muestra** consentida; o estimarlo de forma agregada con cuidado. Muchas regulaciones (incluida la Ley de IA de la UE para sistemas de alto riesgo) permiten tratar datos sensibles con este fin, con garantías.

**B.** Un buen argumento: **TPR iguales** (igualdad de oportunidades). El objetivo es que los pacientes **que lo necesitan** tengan la misma probabilidad de recibir ayuda, sin importar su grupo. Igualar tasas de selección podría dejar fuera a enfermos de un grupo con más enfermedad, o incluir a personas sanas de otro. Lo importante es justificarlo y revisar también el FPR (recursos que se gastan en quien no los necesita).
</details>

### 9.3 🧩 Práctica integradora: "Auditoría al proveedor"

Tu empresa quiere comprar el filtro de CVs de un proveedor. Te dan acceso a sus predicciones sobre **6,000 candidatos** de otra empresa, donde el sesgo de los evaluadores históricos fue menor. Tu trabajo: auditarlo en 5 pasos.

**Paso 1 ▶️ — Datos y modelo del proveedor**

In [ ]:
otra_empresa = generar_candidatos(n=6_000, sesgo_evaluadores=0.4, semilla=7)
modelo_proveedor = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(
    otra_empresa[VARIABLES], otra_empresa.contratado_historico)       # el proveedor entrenó con las etiquetas históricas
auditoria = generar_candidatos(n=6_000, sesgo_evaluadores=0.4, semilla=8)   # candidatos nuevos para auditar
auditoria["seleccionado"] = (modelo_proveedor.predict_proba(auditoria[VARIABLES])[:, 1] > 0.5).astype(int)
print(f"Exactitud que anuncia el proveedor: {(auditoria.seleccionado == auditoria.contratado_historico).mean():.1%}")

**Paso 2 🔮 — Audita.** El sesgo de esa empresa fue menor (0.4 en lugar de 0.7). ¿Pasará la regla 4/5? Ejecuta la auditoría.

In [ ]:
auditar(auditoria)

<details><summary><b>👀 Ver respuesta</b></summary>

Aunque el sesgo histórico fue menor, el modelo **no pasa** la regla 4/5 (impacto dispar ≈ **0.62**) y el TPR del grupo B (≈ 0.51) sigue claramente por debajo del de A (≈ 0.72). Y eso con una exactitud del 83 % que el proveedor anuncia con orgullo. Menos sesgo en los datos = menos sesgo en el modelo, pero **no cero**: el modelo hereda la parte de sesgo que había, más la desigualdad estructural de la experiencia.
</details>

**Paso 3 ✏️ — Busca proxies.** Usa el "adivino" del Ejercicio 5 con estos datos: ¿qué variable delata más el grupo?

In [ ]:
for v in VARIABLES:
    a = make_pipeline(StandardScaler(), LogisticRegression()).fit(auditoria[[v]], auditoria.grupo == "B")
    print(f"{v:<22} → AUC adivinando el grupo: {roc_auc_score(auditoria.grupo == 'B', a.predict_proba(auditoria[[v]])[:, 1]):.3f}")
print("Coeficientes del proveedor:", dict(zip(VARIABLES, modelo_proveedor[-1].coef_[0].round(2))))

<details><summary><b>👀 Ver respuesta</b></summary>

`zona` y `brecha_laboral_meses` son las que más delatan el grupo (AUC ≈ 0.8 cada una), y el modelo del proveedor les da coeficientes **negativos**: está penalizando, en parte, pertenecer al grupo B. Pregunta clave para el proveedor: *¿por qué un filtro de CVs usa la zona donde vive el candidato?*
</details>

**Paso 4 ✏️ — Una contrapropuesta.** Reentrena el modelo del proveedor **sin** `zona` ni `brecha_laboral_meses` (con los datos de `otra_empresa`) y audítalo de nuevo con `auditoria`. ¿Pasa la regla 4/5?

In [ ]:
VARS_LIMPIAS = ["puntaje_prueba", "anios_experiencia"]   # ✏️
m_limpio = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)).fit(
    otra_empresa[VARS_LIMPIAS], otra_empresa.contratado_historico)
a2 = auditoria.copy()
a2["seleccionado"] = (m_limpio.predict_proba(a2[VARS_LIMPIAS])[:, 1] > 0.5).astype(int)
auditar(a2)

<details><summary><b>👀 Ver respuesta</b></summary>

Mejora (impacto dispar de 0.62 a ≈ **0.77**; TPR de B de 0.51 a ≈ 0.58), pero **todavía no llega a 0.8**: quedan el sesgo de la etiqueta y la desigualdad en los años de experiencia. Es una mejora barata, pero no una solución completa.
</details>

**Paso 5 ✋ — Tu recomendación al comité de compras.** 3–4 frases.

✍️ *Mi recomendación:*

<details><summary><b>👀 Ver respuesta</b></summary>

Un ejemplo:

> «No recomiendo comprar el filtro tal como está: selecciona a los candidatos del grupo B en una proporción claramente menor que a los del grupo A (no cumple la regla de los cuatro quintos), aunque su capacidad sea la misma, y usa variables como la zona de residencia que funcionan como sustituto del grupo. Quitando esas variables la brecha se reduce, pero no desaparece, porque el modelo aprendió de decisiones de contratación pasadas. Pediríamos al proveedor entrenar con una medida de desempeño real, auditar por grupo cada trimestre y mantener a una persona revisando a los candidatos descartados mientras tanto.»
</details>

---
## 📚 Para profundizar
- 📘 **Libro (gratis en línea):** Barocas, Hardt y Narayanan, *Fairness and Machine Learning: Limitations and Opportunities*.
- 📰 **Casos:** ProPublica (2016), *Machine Bias* (COMPAS) · Obermeyer et al. (2019), *Dissecting racial bias in an algorithm used to manage the health of populations* · Buolamwini y Gebru (2018), *Gender Shades*.
- 📰 **Imposibilidad:** Kleinberg, Mullainathan y Raghavan (2016), *Inherent Trade-Offs in the Fair Determination of Risk Scores*.
- 🛠️ **Para Colab:** `fairlearn` y AI Fairness 360 (IBM).
- 🎬 **Documental:** *Coded Bias* (2020).

---
⬅️ **Anterior:** [15.3 Reentrenamiento y versionado de datos y modelos](15.3-reentrenamiento-y-versionado.ipynb) · ➡️ **Siguiente:** [15.5 Privacidad, seguridad y ataques adversarios](15.5-privacidad-seguridad-y-ataques-adversarios.ipynb)